In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
import pipeline.src.python.config as cfg
import pandas as pd
import numpy as np
pd.set_option('display.max_colwidth', None)

In [3]:
MAGAZINE_1 = 'the_guardian'
DATASET_TEXT_FEATURE = (
    "text"  # In the dataset file, the column name that contains the text data
)
cfg_dict_1 = cfg.MAGAZINE_CONFIG[MAGAZINE_1]

In [4]:
MAGAZINE_2 = 'scopus'
cfg_dict_2 = cfg.MAGAZINE_CONFIG[MAGAZINE_2]

## Loading models

In [5]:
from bertopic import BERTopic

model_path = cfg.MODELS_FOLDER / f'{MAGAZINE_1}/model_0.302.safetensors'
model_1 = BERTopic.load(model_path, 
                      embedding_model=cfg.EMBEDDING_MODEL
                      )

/home/banfi/.uve/cuda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
model_path_2 = cfg.MODELS_FOLDER / f'{MAGAZINE_2}/model_0.343.safetensors'

model_2 = BERTopic.load(model_path_2,
                        cfg.EMBEDDING_MODEL
                        )


## Loading data

In [7]:
data = np.load(cfg_dict_1['OUTPUT_PATH'],allow_pickle=True) 

ids = data['id']
texts = data['text'] 
embeddings = data['embedding'] 
documents = data['clean_text']

In [8]:
data_2 = np.load(cfg_dict_2['OUTPUT_PATH'],allow_pickle=True) 

ids_2 = data_2['id']
texts_2 = data_2['text'] 
embeddings_2 = data_2['embedding'] 
documents_2 = data_2['clean_text']

## Evaluate metrics

### Model 1 -> Model 2

In [9]:
from pipeline.src.python.btm import BTM

models_metrics = BTM(model_1=model_1,model_2=model_2,ids_1=ids,texts_1=texts,embeddings_1=embeddings)

2026-01-28 14:11:11,952 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


In [10]:
models_metrics.evaluate_metrics()

In [11]:
models_metrics.print_corpus_metrics()

Corpus Closeness: 0.9939046396921348
Corpus Uniqueness: 0.006095360307865127
Corpus Alignment: 0.4087408483016732


In [12]:
models_metrics.print_weighted_corpus_metrics()

Corpus Closeness: 0.9930607808711647
Corpus Uniqueness: 0.0069392191288353815
Corpus Alignment: 0.39316668060084164


In [23]:
a = models_metrics.get_topic_closeness().sort_values(by='Closeness',ascending=False)

In [25]:
len(a[ a['Closeness'] >= 0.5])

57

In [26]:
models_metrics.get_topic_uniqueness().sort_values(by='Closeness',ascending=False).head(10)

,Topic_model_1,Topic_model_2,Couple_counts,Count,Closeness
347,70,-1,18,152,0.118421
562,67,-1,11,154,0.071429
1014,148,-1,5,70,0.071429
707,64,-1,8,161,0.049689
1154,122,-1,4,85,0.047059
874,83,-1,6,130,0.046154
115,1,-1,49,1192,0.041107
1083,89,-1,5,125,0.040000
884,68,-1,6,154,0.038961
445,17,-1,14,402,0.034826


In [17]:
alignment_totale = models_metrics.get_topic_alignment()['Alignment'].sum()

In [18]:
models_metrics.get_topic_closeness()['Topic_model_1'].sort_values().unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28])

In [19]:
alignment_totale / models_metrics.n_topics_model_1

np.float64(0.5480346981532991)

In [20]:
alignment_totale / len(models_metrics.get_topic_alignment())

np.float64(0.5480346981532991)

### Model 2 -> Model 1

In [15]:
inverse_models_metrics = BTM(model_1=model_2,model_2=model_1,ids_1=ids_2,texts_1=texts_2,embeddings_1=embeddings_2)

2026-01-28 14:11:42,144 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


In [16]:
inverse_models_metrics.evaluate_metrics()

In [17]:
inverse_models_metrics.print_corpus_metrics()

Corpus Closeness: 0.9976862503550251
Corpus Uniqueness: 0.002313749644974893
Corpus Alignment: 0.4423351464760438


In [18]:
inverse_models_metrics.print_weighted_corpus_metrics()

Corpus Closeness: 0.9975917584622932
Corpus Uniqueness: 0.002408241537706819
Corpus Alignment: 0.4335854129369716


In [19]:
x = inverse_models_metrics.get_topic_closeness().sort_values(by='Closeness',ascending=False)

In [20]:
x[ x['Closeness'] >= 0.5 ]

,Topic_model_1,Topic_model_2,Couple_counts,Count,Closeness
87,196,35,227,227,1.000000
31,75,72,489,492,0.993902
250,360,147,100,101,0.990099
51,105,149,376,383,0.981723
9,42,57,777,793,0.979823
...,...,...,...,...,...
416,312,32,65,128,0.507812
664,420,21,41,81,0.506173
545,366,149,50,99,0.505051
138,131,35,155,307,0.504886


In [21]:
inverse_models_metrics.get_topic_uniqueness().sort_values(by='Closeness',ascending=False).head(10)

,Topic_model_1,Topic_model_2,Couple_counts,Count,Closeness
2116,293,-1,12,138,0.086957
4100,413,-1,5,84,0.059524
2496,271,-1,9,158,0.056962
2017,186,-1,13,239,0.054393
2443,204,-1,10,222,0.045045
2537,217,-1,9,205,0.043902
4564,373,-1,4,96,0.041667
2594,119,-1,9,340,0.026471
1647,58,-1,16,633,0.025276
5222,325,-1,3,121,0.024793


In [31]:
closeness_matrix = inverse_models_metrics.get_topic_closeness()

In [35]:
closeness_matrix[ closeness_matrix['Topic_model_1'] == 11]

,Topic_model_1,Topic_model_2,Couple_counts,Count,Closeness
12,11,3,320,527,0.607211
124,11,0,60,527,0.113852
125,11,20,60,527,0.113852
387,11,8,18,527,0.034156
477,11,28,14,527,0.026565
642,11,13,9,527,0.017078
872,11,19,6,527,0.011385
1041,11,21,4,527,0.007590
1042,11,24,4,527,0.007590
1099,11,4,4,527,0.007590


### New proposed topic alignment

In [22]:
inverse_alignment_totale = inverse_models_metrics.get_topic_alignment()['Alignment'].sum()

In [23]:
inverse_alignment_totale / inverse_models_metrics.n_topics_model_1

np.float64(0.5991314541264353)

In [24]:
inverse_alignment_totale / len(inverse_models_metrics.get_topic_alignment())

np.float64(0.5991314541264353)

In [25]:
inverse_models_metrics.get_topic_uniqueness().sort_values(by='Closeness',ascending=False).head(10)

,Topic_model_1,Topic_model_2,Couple_counts,Count,Closeness
17,1,-1,13,211,0.061611
24,5,-1,4,77,0.051948
21,0,-1,7,316,0.022152
33,3,-1,2,97,0.020619
54,2,-1,1,105,0.009524
